In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv("powerplant_data.csv")

# AT -> Atmospheric Temperature  
# V -> Vacuum
# AP ->  Atmoshperic Pressure 
# RH -> Relative Humidity

# PE -> Produced Energy


In [ ]:
X = df.drop(columns=["PE"])
y = df["PE"]

In [ ]:
from sklearn.model_selection import train_test_split

X_train , X_test , y_train , y_test = train_test_split(
    X,y , random_state=42, test_size= 0.2
)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
import torch 
import torch.nn as nn

X_train_tensor = torch.tensor(X_train_scaled,dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled,dtype=torch.float32)

y_train_tensors = torch.tensor(y_train.values,dtype=torch.float32).view(-1,1)
y_test_tensors = torch.tensor(y_test.values,dtype=torch.float32).view(-1,1)

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

train_dataset = TensorDataset(X_train_tensor,y_train_tensors)
test_dataset = TensorDataset(X_test_tensor,y_test_tensors)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32,shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

In [ ]:
# Define ANN model 

class ANN(nn.Module):
    def __init__(self,):
        super(ANN,self).__init__()

        self.model = nn.Sequential(
            # 1st hidden Layer
            nn.Linear(X_train.shape[1],6),
            nn.ReLU(),

            # 2nd hidden Layer
            nn.Linear(6,6),
            nn.ReLU(),

            # o/p Layer
            nn.Linear(6,1)     
        )

    def forward(self,x):
        return self.model(x)

In [ ]:
import torch.optim as optim 

model = ANN()

# loss optimizer 
crietrion = nn.MSELoss()
optimizer = optim.Adam(model.parameters())

In [ ]:
# Train the ANN 

train_lossses = []
val_losses = []
epochs = 100

best_val_loss = float("inf")

for epoch in range(epochs):
    model.train()
    running_loss = 0.0 # for training loss for 1 epoch

    for xb,yb in train_loader:
        #xb =features of 1 batch
        #yb =labels of 1 batch
        optimizer.zero_grad()

        outputs = model(xb) #forward prop ..predicted o/p
        loss = crietrion(outputs,yb) # compute loss
        loss.backward() # backward prop ..compute GD
        optimizer.step() # params update

        running_loss += loss.item() # loss is tensor so -> convert to float
    epoch_train_loss = running_loss/len(train_loader)
    train_lossses.append(epoch_train_loss)

    # validation 
    model.eval()
    running_val_loss = 0.0
    with torch.no_grad(): # no GD Compute
        for xb,yb in test_loader:
            outputs = model(xb)
            loss = crietrion(outputs,yb)
            running_val_loss += loss

    epoch_val_loss = running_val_loss/len(test_loader)
    val_losses.append(epoch_val_loss)

    print(f"epoch {epoch+1}/{epochs} ==> train loss = {epoch_train_loss} & val loss = {epoch_val_loss}")

    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(),"best_model.pt")

In [ ]:
import matplotlib.pyplot as plt 

loss_df = pd.DataFrame({
    "Training Loss": train_lossses,
    "Validation Loss": val_losses
})

plt.plot(loss_df["Training Loss"],label="Training Loss")
plt.plot(loss_df["Validation Loss"],label="Validation Loss")
plt.xlabel("Epochs")
plt.ylabel("Losses")
plt.legend()

In [ ]:
model.load_state_dict(torch.load("best_model.pt"))

In [ ]:
# Evalute 

model.eval()
with torch.no_grad():
    train_preds = model(X_train_tensor)
    test_preds = model(X_test_tensor)

    train_mse_loss = crietrion(train_preds,y_train_tensors)
    test_mse_loss = crietrion(test_preds,y_test_tensors)

print("Training MSE",train_mse_loss.item())
print("Testing MSE",test_mse_loss.item())

In [ ]:
from sklearn.metrics import r2_score

print("r2 score: ",r2_score(y_test,test_preds))

In [ ]:
predicted_df = pd.DataFrame(test_preds.numpy(),columns= ["Predicted values"])
actutal_df = pd.DataFrame(y_test.numpy(),columns= ["Actual values"])

pd.concat([predicted_df,actutal_df],axis=1)